# COMP6011 — LLM-Based Suicide Risk Detection Benchmark
**RTX 3060 12GB | C-SSRS Framework | SAK-aligned output**

## Models
| # | Model | Type | Reference |
|---|-------|------|-----------|
| 1 | MentalRoBERTa (NLI zero-shot) | Encoder discriminative baseline | Ji et al. (2022) |
| 2 | MentaLLaMA-chat-7B | Specialized mental health LLM | Yang et al. (2024) |
| 3 | Mistral-7B-Instruct-v0.2 | General LLM + C-SSRS prompt | Patil et al. (2025) |
| 4 | Llama-3.1-8B-Instruct | Open-source LLM on-premise | Meta (2024) |

## Datasets
| # | Dataset | Type | N |
|---|---------|------|---|
| 1 | Reddit C-SSRS (Gaur et al. 2019) | Social media posts | 100 |
| 2 | Human-AI Dialogs (Zenodo 2026) | Multi-turn dialogues | 100 |
| 3 | Assignment 10-cases (Professor) | Clinical dialogues PoC | 10 |

## Key evaluation metrics
- **Recall High-Risk (≥4)** — most clinically critical: false negatives cost lives
- **MAE Ordinal** — measures severity of errors (off-by-1 vs off-by-5)
- **Macro F1** — balanced across all risk classes
- **Adjacent Accuracy (±1)** — clinically acceptable predictions
- **Parse Rate** — LLM output reliability

## C-SSRS → SAK mapping (Australia)
Benchmarking uses C-SSRS (0-6) as it matches available annotated datasets.
Output for clinicians is mapped to Australian SAK categories:
- C-SSRS 0-2 → **Low Risk** (SAK)
- C-SSRS 3   → **Moderate Risk** (SAK)
- C-SSRS 4-5 → **High Risk** (SAK)
- C-SSRS 6   → **Imminent Risk** (SAK)

---
**Execution order:** Run cells top to bottom. Results are auto-saved to Excel after every dataset so no progress is lost if the session disconnects.

In [ ]:
# ================================================================
# CELL 1 — Install dependencies
# Run once. Safe to re-run — skips already-installed packages.
# ================================================================
import subprocess, sys

PACKAGES = [
    'transformers>=4.40.0',
    'bitsandbytes',          # 4-bit quantisation
    'accelerate',            # device_map support
    'datasets',              # HuggingFace datasets
    'scikit-learn',          # metrics
    'pandas',
    'openpyxl',              # read/write .xlsx
    'tqdm',
    'sentencepiece',         # tokenizers for some models
    'seaborn',
    'matplotlib',
    'xlsxwriter',            # formatted Excel output
]

for pkg in PACKAGES:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        capture_output=True, text=True
    )
    status = '✅' if result.returncode == 0 else '❌'
    print(f'{status} {pkg}')

print('\nAll dependencies ready.')

In [ ]:
# ================================================================
# CELL 2 — Imports and GPU verification
# ================================================================
import os, gc, json, re, time, ast, warnings
from pathlib import Path

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from tqdm import tqdm
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, pipeline
)
from sklearn.metrics import (
    f1_score, recall_score, precision_score,
    mean_absolute_error, confusion_matrix,
    classification_report, cohen_kappa_score,
    roc_auc_score
)
from scipy.stats import spearmanr

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 130

# ── GPU check ────────────────────────────────────────────────
assert torch.cuda.is_available(), (
    'CUDA not found. Make sure you selected the correct Python kernel '
    'with PyTorch+CUDA. In VSCode: Ctrl+Shift+P → Jupyter: Select Kernel'
)

GPU_NAME   = torch.cuda.get_device_name(0)
VRAM_TOTAL = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f'GPU:          {GPU_NAME}')
print(f'VRAM total:   {VRAM_TOTAL:.1f} GB')
print(f'CUDA version: {torch.version.cuda}')
print(f'PyTorch:      {torch.__version__}')

def vram_status(label: str = ''):
    used = torch.cuda.memory_allocated(0) / 1e9
    free = VRAM_TOTAL - used
    tag  = f' [{label}]' if label else ''
    print(f'  VRAM{tag}: {used:.1f} GB used / {free:.1f} GB free')

In [ ]:
# ================================================================
# CELL 3 — Path configuration
# ── EDIT THE PATHS BELOW TO MATCH YOUR LOCAL FILES ───────────
# ================================================================

# Directory where all results (Excel + PNG) will be saved.
# Created automatically if it does not exist.
RESULTS_DIR = Path('./results')

# ── Local dataset files ───────────────────────────────────────
# Reddit C-SSRS: the TSV file downloaded from Zenodo.
# Columns expected: User | Post | Label
REDDIT_PATH = Path('./datasets/500_Reddit_users_posts_labels.csv')

# Human-AI Dialogs (Zenodo 2026): Excel or CSV.
# Columns expected: index | dialogue | post_risk
HUMAN_AI_PATH = Path('./datasets/Human-AI Dialogue Suicide Risk Dataset_4040_Release.xlsx')

# 10-case assignment dataset from your professor.
# Columns expected: Case ID | Paraphrased Dialogue | Risk Level
CASES_PATH    = Path('./datasets/student_assignment_10_cases.xlsx')

# MHDialog: multi-turn mental health dialogues with risk labels
# Downloaded from HuggingFace (IkeZhang/MHDialog) and saved locally.
MHDIALOG_PATH = Path('./datasets/mhdialog_full.csv')

# ── HuggingFace model cache (change if disk space is limited) ─
# Uncomment and set a path on a disk with ≥50 GB free space.
# os.environ['HF_HOME'] = '/path/to/large/disk/.cache/huggingface'

# ── Sample size per dataset ──────────────────────────────────
# 30  = quick validation (~40 min, 6 models x 4 datasets)
# 50  = balanced benchmark (~90 min)  <- recommended
# 100 = full benchmark (~3 hours)
N_SAMPLES = 10

# ── Column name mappings — adjust to match your actual files ──
# Reddit C-SSRS
REDDIT_TEXT_COL  = 'Post'
REDDIT_LABEL_COL = 'Label'

# Human-AI Dialogs (Zenodo)
HUMAN_AI_TEXT_COL  = 'dialogue'
HUMAN_AI_LABEL_COL = 'post_risk'

# 10-case dataset
CASES_ID_COL    = 'Case ID'
CASES_TEXT_COL  = 'Paraphrased Dialogue'
CASES_LABEL_COL   = 'Risk Level'

# MHDialog column mappings
MHDIALOG_TEXT_COL  = 'text'
MHDIALOG_LABEL_COL = 'concern_type'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Results directory: {RESULTS_DIR.resolve()}')
print(f'Sample size per dataset: N={N_SAMPLES}')


In [ ]:
# ================================================================
# CELL 4 — Load and prepare all four datasets
# ================================================================

# ─────────────────────────────────────────────────────────────
# DATASET 1: Reddit C-SSRS
# Posts annotated by four psychiatrists using C-SSRS guidelines.
# Reference: Gaur et al. (2019), Zenodo.
# Labels (Title Case): Supportive | Indicator | Ideation | Behavior | Attempt
# ─────────────────────────────────────────────────────────────
print('─' * 55)
print('Loading Dataset 1: Reddit C-SSRS')

df_reddit_raw = pd.read_csv(REDDIT_PATH, sep=None, engine='python')
print(f'  Shape:   {df_reddit_raw.shape}')
print(f'  Columns: {df_reddit_raw.columns.tolist()}')
print(f'  Labels:  {df_reddit_raw[REDDIT_LABEL_COL].value_counts().to_dict()}')

# Posts are stored as Python list-of-strings — parse and join
def parse_posts(raw: str, max_chars: int = 3000) -> str:
    try:
        posts = ast.literal_eval(str(raw))
        return ' '.join(str(p) for p in posts)[:max_chars]
    except Exception:
        return str(raw)[:max_chars]

df_reddit_raw['text'] = df_reddit_raw[REDDIT_TEXT_COL].apply(parse_posts)

# Map string labels → C-SSRS integer (0-6)
# Exact labels present in file (Title Case, confirmed):
#   Supportive(108), Indicator(99), Ideation(171), Behavior(77), Attempt(45)
REDDIT_LABEL_MAP = {
    'Supportive': 0,
    'Indicator':  1,
    'Ideation':   2,
    'Behavior':   4,
    'Attempt':    5,
}
df_reddit_raw['true_label'] = df_reddit_raw[REDDIT_LABEL_COL].map(REDDIT_LABEL_MAP)
df_reddit_raw = df_reddit_raw.dropna(subset=['true_label'])
df_reddit_raw['true_label'] = df_reddit_raw['true_label'].astype(int)

# Stratified sample to balance classes
n_classes_r  = df_reddit_raw['true_label'].nunique()
n_per_class  = max(1, N_SAMPLES // n_classes_r)
df_reddit = (
    df_reddit_raw
    .groupby('true_label', group_keys=False)
    .apply(lambda x: x.sample(min(len(x), n_per_class), random_state=42))
    .reset_index(drop=True)
)
print(f'  Sampled: {len(df_reddit)} rows')
print(f'  Distribution: {df_reddit["true_label"].value_counts().sort_index().to_dict()}')

# ─────────────────────────────────────────────────────────────
# DATASET 2: Human-AI Dialogs (Zenodo 2026)
# Multi-turn human-AI dialogues annotated with post_risk labels.
# Labels (lowercase, confirmed): safe | indicator | ideation | behavior | attempt
# ─────────────────────────────────────────────────────────────
print('\n' + '─' * 55)
print('Loading Dataset 2: Human-AI Dialogs (Zenodo 2026)')

if HUMAN_AI_PATH.suffix == '.xlsx':
    df_humanai_raw = pd.read_excel(HUMAN_AI_PATH)
else:
    df_humanai_raw = pd.read_csv(HUMAN_AI_PATH)

print(f'  Shape:   {df_humanai_raw.shape}')
print(f'  Columns: {df_humanai_raw.columns.tolist()}')
print(f'  Labels:  {df_humanai_raw[HUMAN_AI_LABEL_COL].value_counts().to_dict()}')

df_humanai_raw['text'] = df_humanai_raw[HUMAN_AI_TEXT_COL].astype(str).str[:3000]

# Exact labels present in file (lowercase, confirmed):
#   safe(2040), indicator(783), ideation(759), behavior(381), attempt(77)
HUMANAI_LABEL_MAP = {
    'safe':      0,
    'indicator': 1,
    'ideation':  2,
    'behavior':  4,
    'attempt':   5,
}
df_humanai_raw['true_label'] = (
    df_humanai_raw[HUMAN_AI_LABEL_COL]
    .astype(str).str.lower().str.strip()
    .map(HUMANAI_LABEL_MAP)
)

unmapped = df_humanai_raw[df_humanai_raw['true_label'].isna()][HUMAN_AI_LABEL_COL].unique()
if len(unmapped) > 0:
    print(f'  ⚠️  Unmapped labels (add to HUMANAI_LABEL_MAP): {unmapped}')

df_humanai_raw = df_humanai_raw.dropna(subset=['true_label'])
df_humanai_raw['true_label'] = df_humanai_raw['true_label'].astype(int)

n_classes_h = df_humanai_raw['true_label'].nunique()
n_per_class_h = max(1, N_SAMPLES // n_classes_h)
df_humanai = (
    df_humanai_raw
    .groupby('true_label', group_keys=False)
    .apply(lambda x: x.sample(min(len(x), n_per_class_h), random_state=42))
    .reset_index(drop=True)
)
print(f'  Sampled: {len(df_humanai)} rows')
print(f'  Distribution: {df_humanai["true_label"].value_counts().sort_index().to_dict()}')

# ─────────────────────────────────────────────────────────────
# DATASET 3: 10-case assignment dataset (PoC)
# Exact labels present (lowercase, confirmed):
#   safe(2), indicator(2), ideation(2), behavior(2), attempt(2)
# ─────────────────────────────────────────────────────────────
print('\n' + '─' * 55)
print('Loading Dataset 3: 10-case Assignment Dataset')

df_raw_cases = pd.read_excel(CASES_PATH, sheet_name='Assignment_Cases')
print(f'  Columns: {df_raw_cases.columns.tolist()}')

CASES_LABEL_MAP = {
    'safe':      0,
    'indicator': 1,
    'ideation':  2,
    'behavior':  4,
    'attempt':   5,
}

case_rows = []
for cid in df_raw_cases[CASES_ID_COL].unique():
    rows       = df_raw_cases[df_raw_cases[CASES_ID_COL] == cid]
    text       = ' '.join(rows[CASES_TEXT_COL].astype(str).tolist())[:3000]
    label_str  = str(rows[CASES_LABEL_COL].iloc[0])
    label_num  = CASES_LABEL_MAP.get(label_str.lower().strip(), -1)
    case_rows.append({
        'case_id':        cid,
        'text':           text,
        'true_label_str': label_str,
        'true_label':     label_num
    })

df_10cases = pd.DataFrame(case_rows)
df_10cases = df_10cases[df_10cases['true_label'] != -1].reset_index(drop=True)
unmapped_cases = [r['true_label_str'] for r in case_rows if r['true_label'] == -1]
if unmapped_cases:
    print(f'  ⚠️  Unmapped case labels (add to CASES_LABEL_MAP): {unmapped_cases}')

print(f'  Cases loaded: {len(df_10cases)}')
print(df_10cases[['case_id', 'true_label_str', 'true_label']])


# ─────────────────────────────────────────────────────────────
# DATASET 4: MHDialog (IkeZhang/MHDialog — HuggingFace)
# Exact labels present (Title Case, confirmed):
#   Not Related(569), Ideation(241), Behavior(100), Unsure(36),
#   Attempt(23), Supportive(19), Indicator(12)
# 'Unsure' has no C-SSRS equivalent and is dropped.
# ─────────────────────────────────────────────────────────────
print('\n' + '─' * 55)
print('Loading Dataset 4: MHDialog (IkeZhang/HuggingFace)')

df_mhdialog_raw = pd.read_csv(MHDIALOG_PATH)
print(f'  Shape:   {df_mhdialog_raw.shape}')
print(f'  Labels:  {df_mhdialog_raw[MHDIALOG_LABEL_COL].value_counts().to_dict()}')

MHDIALOG_LABEL_MAP = {
    'Not Related': 0,
    'Supportive':  0,
    'Indicator':   1,
    'Ideation':    2,
    'Behavior':    4,
    'Attempt':     5,
    # 'Unsure' intentionally omitted — no C-SSRS equivalent
}
df_mhdialog_raw['true_label'] = (
    df_mhdialog_raw[MHDIALOG_LABEL_COL]
    .astype(str).str.strip()
    .map(MHDIALOG_LABEL_MAP)
)
df_mhdialog_raw = df_mhdialog_raw.dropna(subset=['true_label'])
df_mhdialog_raw['true_label'] = df_mhdialog_raw['true_label'].astype(int)

n_classes_mh   = df_mhdialog_raw['true_label'].nunique()
n_per_class_mh = max(1, N_SAMPLES // n_classes_mh)
df_mhdialog = (
    df_mhdialog_raw
    .groupby('true_label', group_keys=False)
    .apply(lambda x: x.sample(min(len(x), n_per_class_mh), random_state=42))
    .reset_index(drop=True)
)
print(f'  Sampled: {len(df_mhdialog)} rows')
print(f'  Distribution: {df_mhdialog["true_label"].value_counts().sort_index().to_dict()}')
print('\n✅ All datasets ready')

In [ ]:
# ================================================================
# CELL 5 — Shared prompt, helpers, Excel saver, inference runner
# ================================================================

# ─── C-SSRS clinical prompt (identical for every model) ────────
# SAK mapping intentionally excluded from this prompt.
# Benchmark evaluates C-SSRS ordinal classification only.
# SAK mapping is applied deterministically in build_result_row via
# cssrs_to_sak() — models are not asked to produce SAK categories.

SYSTEM_PROMPT = (
    'You are a clinical AI assistant trained to assess suicide risk '
    'using the Columbia Suicide Severity Rating Scale (C-SSRS). '
    'Output ONLY valid JSON with no preamble, no markdown, no explanation.'
)

USER_TEMPLATE = """Analyze the text below and provide a structured suicide risk assessment.

C-SSRS Risk Levels:
  0 = Safe            — no suicidal ideation or self-harm indicators
  1 = Passive ideation — wishes to be dead; no intent or plan
  2 = Active ideation, no plan — suicidal thoughts; no specific method
  3 = Active ideation with plan — specific method or timing considered
  4 = Preparatory behaviour — concrete steps toward an attempt
  5 = Past attempt / severe distress — history of attempt or extreme pain
  6 = Imminent risk — immediate danger or active self-harm

TEXT TO ANALYSE:
{text}

Respond ONLY with this JSON object (no other text):
{{
  "cssrs_level": <integer 0-6>,
  "confidence": "<low|medium|high>",
  "key_signals": ["<signal1>", "<signal2>"],
  "justification": "<2-3 sentence clinical reasoning>"
}}"""


# ─── Helper functions ──────────────────────────────────────────

def cssrs_to_sak(level: int) -> str:
    """Map C-SSRS level to Australian SAK category."""
    if level <= 2:   return 'Low Risk'
    elif level == 3: return 'Moderate Risk'
    elif level <= 5: return 'High Risk'
    else:            return 'Imminent Risk'

def score_to_conf(s: float) -> str:
    return 'high' if s >= 0.7 else 'medium' if s >= 0.4 else 'low'


def parse_llm_response(text: str) -> dict:
    """
    Extract structured data from LLM output.
    Handles: IBM Granite <thinking> tokens, DeepSeek/Llama <think> tokens,
    markdown fences, partial JSON.
    Returns a dict with cssrs_level = -1 if parsing fails completely.
    """
    # Strip chain-of-thought tokens
    if '</thinking>' in text:          # IBM Granite 3.2 thinking tokens
        text = text.split('</thinking>')[-1].strip()
    elif '</think>' in text:           # DeepSeek / Qwen3 think tokens
        text = text.split('</think>')[-1].strip()
    # Strip markdown code fences
    text = re.sub(r'```(?:json)?\s*', '', text).strip()

    # Attempt 1: full valid JSON block
    try:
        m = re.search(r'\{[^{}]*\}', text, re.DOTALL)
        if m:
            d = json.loads(m.group())
            if 'cssrs_level' in d:
                d['cssrs_level'] = int(d['cssrs_level'])
                return d
    except Exception:
        pass

    # Attempt 2: extract cssrs_level value only
    try:
        m = re.search(r'["\']?cssrs_level["\']?\s*:\s*(\d)', text)
        if m:
            level = int(m.group(1))
            j = re.search(r'["\']?justification["\']?\s*:\s*["\']([^"\'\n]{10,})', text)
            return {
                'cssrs_level':  level,
                'confidence':   'low',
                'key_signals':  [],
                'justification': j.group(1)[:200] if j else 'Partial parse'
            }
    except Exception:
        pass

    # Attempt 3: any lone digit 0-6 in the output
    try:
        m = re.search(r'\b([0-6])\b', text)
        if m:
            level = int(m.group(1))
            return {
                'cssrs_level':  level,
                'confidence':   'low',
                'key_signals':  [],
                'justification': f'Fallback parse from: {text[:150]}'
            }
    except Exception:
        pass

    # Complete failure
    return {
        'cssrs_level': -1,
        'confidence': 'none', 'key_signals': [],
        'justification': f'Parse failed. Raw: {text[:120]}'
    }


def save_to_excel(df: pd.DataFrame, model_name: str, dataset_name: str):
    """Save result DataFrame to Excel after every model × dataset run."""
    path = RESULTS_DIR / f'{model_name}_{dataset_name}.xlsx'
    with pd.ExcelWriter(path, engine='xlsxwriter') as writer:
        df.to_excel(writer, sheet_name='Results', index=False)
        ws = writer.sheets['Results']
        for i, col in enumerate(df.columns):
            max_w = max(df[col].astype(str).map(len).max(), len(col)) + 2
            ws.set_column(i, i, min(max_w, 80))
    print(f'  💾 Saved → {path.name}')


def load_if_saved(model_name: str, dataset_name: str):
    """Load previously saved results to skip re-running a completed combination."""
    path = RESULTS_DIR / f'{model_name}_{dataset_name}.xlsx'
    if path.exists():
        print(f'  📂 Loaded existing results: {path.name}')
        return pd.read_excel(path)
    return None


def build_result_row(
    dataset_name, model_name, idx, true_label,
    parsed, latency_s
) -> dict:
    """Construct a standardised result row for any model."""
    pred     = parsed.get('cssrs_level', -1)
    parse_ok = pred != -1
    return {
        'dataset':       dataset_name,
        'model':         model_name,
        'idx':           idx,
        'true_label':    true_label,
        'true_sak':      cssrs_to_sak(true_label),
        'pred_label':    pred,
        'pred_sak':      cssrs_to_sak(pred) if parse_ok else 'Parse Error',
        'confidence':    parsed.get('confidence', ''),
        'key_signals':   str(parsed.get('key_signals', [])),
        'justification': str(parsed.get('justification', ''))[:350],
        'ordinal_error': abs(true_label - pred) if parse_ok else -1,
        'hr_true':       1 if true_label >= 4 else 0,
        'hr_pred':       1 if pred >= 4 else 0,
        'parse_ok':      parse_ok,
        'latency_s':     round(latency_s, 2)
    }


def run_generative_model(
    df, text_col, label_col,
    model, tokenizer,
    model_name, dataset_name,
    use_chat_template=False,
    max_input_chars=2500,
    max_new_tokens=350,
    autosave_every=10
) -> pd.DataFrame:
    """
    Generic inference loop for any generative (decoder-only) model.
    - Loads existing results and skips if already completed.
    - Auto-saves to Excel every `autosave_every` rows.
    - Handles chat template format or plain prompt.
    """
    cached = load_if_saved(model_name, dataset_name)
    if cached is not None:
        return cached

    tmp_path = RESULTS_DIR / f'{model_name}_{dataset_name}_TEMP.xlsx'
    results  = []
    print(f'\n  ▶ {model_name} on {dataset_name} (N={len(df)})...')

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        text       = str(row[text_col])[:max_input_chars]
        true_label = int(row[label_col])
        full_text  = f'{SYSTEM_PROMPT}\n\n{USER_TEMPLATE.format(text=text)}'

        t0 = time.time()
        try:
            if use_chat_template:
                messages = [{'role': 'user', 'content': full_text}]
                prompt   = tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True
                )
            else:
                prompt = full_text

            inputs = tokenizer(
                prompt, return_tensors='pt',
                truncation=True, max_length=3000
            ).to(model.device)

            with torch.no_grad():
                out_ids = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    temperature=None,
                    top_p=None,
                    pad_token_id=tokenizer.eos_token_id
                )

            raw    = tokenizer.decode(
                out_ids[0][inputs['input_ids'].shape[1]:],
                skip_special_tokens=True
            ).strip()
            parsed = parse_llm_response(raw)

        except Exception as e:
            parsed = {
                'cssrs_level': -1,
                'confidence': 'none', 'key_signals': [],
                'justification': f'Runtime error: {str(e)[:100]}'
            }

        results.append(
            build_result_row(
                dataset_name, model_name, idx,
                true_label, parsed,
                time.time() - t0
            )
        )

        if len(results) % autosave_every == 0:
            pd.DataFrame(results).to_excel(tmp_path, index=False)

    result_df = pd.DataFrame(results)
    save_to_excel(result_df, model_name, dataset_name)

    if tmp_path.exists():
        tmp_path.unlink()

    return result_df


def unload_model(model, tokenizer=None, label: str = ''):
    """Delete model from VRAM and run garbage collection."""
    del model
    if tokenizer is not None:
        del tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    vram_status(f'after unloading {label}')


# 4-bit quantisation config shared by all generative models
BNB_4BIT = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True
)

# Dataset list iterated by every model's inference loop
DATASETS = [
    (df_reddit,   'text', 'true_label', 'Reddit_CSSRS'),
    (df_humanai,  'text', 'true_label', 'HumanAI_Dialogs'),
    (df_10cases,  'text', 'true_label', '10_Cases'),
    (df_mhdialog, 'text', 'true_label', 'MHDialog'),
]

# C-SSRS label names for plots
CSSRS_NAMES = {
    0: 'Safe', 1: 'Passive', 2: 'Active\n(NP)',
    3: 'Active\n(P)', 4: 'Prep', 5: 'Past', 6: 'Immin.'
}

print('✅ All helpers and config ready. Proceed to Model 1.')

---
## Model 1 — MentalRoBERTa (Zero-Shot NLI)
**Paradigm:** Encoder-only discriminative baseline — no text generation, no explanation  
**Reference:** Ji et al. (2022) MentalBERT/MentalRoBERTa — LREC 2022  
**Expected time on RTX 3060:** ~8-12 min for all 3 datasets

In [ ]:
# ================================================================
# CELL 6 — Model 1: MentalRoBERTa — Load
# Uses mental/mental-roberta-base: RoBERTa pre-trained on 60K mental
# health Reddit posts (Ji et al. 2022, LREC). No NLI head — classified
# via cosine similarity between text CLS embedding and C-SSRS label
# description embeddings (bi-encoder zero-shot approach).
# First run downloads ~500 MB.
# ================================================================
print('Loading MentalRoBERTa (mental/mental-roberta-base)...')
print('First run downloads ~500 MB...')

import os
os.environ['HF_TOKEN'] = os.environ.get('HF_TOKEN', '')  # set HF_TOKEN in your environment; never hardcode a token

from transformers import AutoModel
import torch.nn.functional as F

mr_tokenizer = AutoTokenizer.from_pretrained('mental/mental-roberta-base')
mr_model     = AutoModel.from_pretrained('mental/mental-roberta-base').to('cuda:0')
mr_model.eval()

# C-SSRS label descriptions — one per level (0-6)
CSSRS_HYPOTHESES = [
    'Safe: no suicide risk indicators present in this text',           # 0
    'Passive ideation: person wishes to be dead but has no intent',    # 1
    'Active suicidal ideation without a specific plan or method',      # 2
    'Active suicidal ideation with a specific plan or method',         # 3
    'Preparatory behavior: concrete steps toward a suicide attempt',   # 4
    'Past suicide attempt or currently experiencing severe distress',  # 5
    'Imminent suicide risk: immediate danger or active self-harm',     # 6
]

def get_cls_embedding(text: str) -> torch.Tensor:
    """Return CLS token embedding from MentalRoBERTa."""
    enc = mr_tokenizer(
        text, return_tensors='pt',
        truncation=True, max_length=512
    ).to(mr_model.device)
    with torch.no_grad():
        return mr_model(**enc).last_hidden_state[:, 0, :]  # [1, hidden_size]

# Pre-compute label embeddings once at load time
label_embs = torch.cat([get_cls_embedding(h) for h in CSSRS_HYPOTHESES])  # [7, hidden_size]

def predict_mental_roberta(text: str) -> dict:
    """Classify text into C-SSRS levels via cosine similarity to label embeddings."""
    text_emb = get_cls_embedding(str(text)[:1500])
    sims     = F.cosine_similarity(text_emb, label_embs)  # [7]
    level    = sims.argmax().item()
    score    = sims[level].item()
    return {
        'cssrs_level':   level,
        'confidence':    score_to_conf(score),
        'key_signals':   [],
        'justification': f'Cosine sim={score:.3f}; nearest label: {CSSRS_HYPOTHESES[level][:70]}'
    }

vram_status('MentalRoBERTa loaded')
print('✅ MentalRoBERTa ready')

In [ ]:
# ================================================================
# CELL 7 — Model 1: MentalRoBERTa — Inference on all datasets
# Results saved to Excel after each dataset.
# ================================================================
mr_results_list = []

for df, text_col, label_col, ds_name in DATASETS:
    cached = load_if_saved('MentalRoBERTa', ds_name)
    if cached is not None:
        mr_results_list.append(cached)
        continue

    print(f'\n  ▶ MentalRoBERTa on {ds_name} (N={len(df)})...')
    rows = []
    t_ds = time.time()

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        true_label = int(row[label_col])
        t0     = time.time()
        parsed = predict_mental_roberta(str(row[text_col]))
        rows.append(
            build_result_row(
                ds_name, 'MentalRoBERTa', idx,
                true_label, parsed,
                time.time() - t0
            )
        )

    df_res = pd.DataFrame(rows)
    save_to_excel(df_res, 'MentalRoBERTa', ds_name)
    mr_results_list.append(df_res)
    print(f'  ✅ {ds_name} done in {time.time()-t_ds:.0f}s')

# Unload from GPU — free VRAM for next model
del mr_model, mr_tokenizer, label_embs
gc.collect()
torch.cuda.empty_cache()
vram_status('after MentalRoBERTa')
print('\n✅ MentalRoBERTa complete on all datasets')

In [ ]:
# ================================================================
# CELL 6 — Model 1: MentalRoBERTa — Load
# Uses cross-encoder/nli-deberta-v3-large as the NLI backbone,
# which provides strong zero-shot classification when paired with
# C-SSRS-derived hypothesis labels.
# ================================================================
print('Loading MentalRoBERTa ZS (NLI backbone: DeBERTa-v3-large)...')
print('First run downloads ~1.5 GB...')

zs_pipe = pipeline(
    'zero-shot-classification',
    model='cross-encoder/nli-deberta-v3-large',
    device=0
)

# One hypothesis per C-SSRS level.
# Phrasing matches clinical language — shown to improve NLI accuracy.
CSSRS_HYPOTHESES = [
    'Safe: no suicide risk indicators present in this text',           # 0
    'Passive ideation: person wishes to be dead but has no intent',    # 1
    'Active suicidal ideation without a specific plan or method',      # 2
    'Active suicidal ideation with a specific plan or method',         # 3
    'Preparatory behavior: concrete steps toward a suicide attempt',   # 4
    'Past suicide attempt or currently experiencing severe distress',  # 5
    'Imminent suicide risk: immediate danger or active self-harm',     # 6
]
ZS_TEMPLATE = 'The suicide risk level of this text is: {}'

def predict_zeroshot_nli(text: str) -> dict:
    """Classify text into C-SSRS levels via NLI zero-shot."""
    result = zs_pipe(
        str(text)[:1500],          # truncate to NLI token limit
        CSSRS_HYPOTHESES,
        hypothesis_template=ZS_TEMPLATE,
        multi_label=False
    )
    level = CSSRS_HYPOTHESES.index(result['labels'][0])
    score = result['scores'][0]
    return {
        'cssrs_level':  level,
        'sak_category': cssrs_to_sak(level),
        'confidence':   score_to_conf(score),
        'key_signals':  [],    # NLI does not extract signal phrases
        'justification': f'NLI score={score:.3f}; winning hypothesis: {CSSRS_HYPOTHESES[level][:70]}'
    }

vram_status('MentalRoBERTa loaded')
print('✅ MentalRoBERTa ZS ready')

In [ ]:
# ================================================================
# CELL 7 — Model 1: MentalRoBERTa — Inference on all 3 datasets
# Results saved to Excel after each dataset.
# ================================================================
mr_results_list = []

for df, text_col, label_col, ds_name in DATASETS:
    # Check if already completed
    cached = load_if_saved('MentalRoBERTa', ds_name)
    if cached is not None:
        mr_results_list.append(cached)
        continue

    print(f'\n  ▶ MentalRoBERTa on {ds_name} (N={len(df)})...')
    rows   = []
    t_ds   = time.time()

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        true_label = int(row[label_col])
        t0     = time.time()
        parsed = predict_zeroshot_nli(str(row[text_col]))
        rows.append(
            build_result_row(
                ds_name, 'MentalRoBERTa', idx,
                true_label, parsed,
                time.time() - t0
            )
        )

    df_res = pd.DataFrame(rows)
    save_to_excel(df_res, 'MentalRoBERTa', ds_name)
    mr_results_list.append(df_res)
    print(f'  ✅ {ds_name} done in {time.time()-t_ds:.0f}s')

# Unload from GPU — free VRAM for next model
del zs_pipe
gc.collect()
torch.cuda.empty_cache()
vram_status('after MentalRoBERTa')
print('\n✅ MentalRoBERTa complete on all datasets')

In [ ]:
gc.collect()
torch.cuda.empty_cache()
vram_status('after MentalRoBERTa')

In [ ]:
unload_model(ml_model, ml_tokenizer, 'MentalRoBERTa')   

---
## Model 2 — MentaLLaMA-chat-7B
**Paradigm:** First open-source LLM purpose-built for mental health — generates classification + explanation  
**Reference:** Yang et al. (2024), WWW'24 — fine-tuned on 105K mental health samples  
**Expected time on RTX 3060:** ~45-60 min for all 3 datasets

In [ ]:
# ================================================================
# CELL 8 — Model 2: MentaLLaMA — Load (4-bit quantised)
# First run downloads ~13 GB from HuggingFace Hub.
# Subsequent runs load from local cache (~5 min).
# ================================================================
print('Loading MentaLLaMA-chat-7B (4-bit NF4 quantisation)...')
print('First run: downloading ~13 GB — may take 10-20 min...')

ml_tokenizer = AutoTokenizer.from_pretrained('klyang/MentaLLaMA-chat-7B')
ml_model     = AutoModelForCausalLM.from_pretrained(
    'klyang/MentaLLaMA-chat-7B',
    quantization_config=BNB_4BIT,
    device_map='cuda:0',
    low_cpu_mem_usage=True
)
ml_model.eval()

vram_status('MentaLLaMA loaded')
print('✅ MentaLLaMA-chat-7B ready')

In [ ]:
if not ml_tokenizer.chat_template:
    ml_tokenizer.chat_template = (
        "{% for message in messages %}"
        "{% if message['role'] == 'user' %}"
        "{{ '<s>[INST] ' + message['content'].strip() + ' [/INST]' }}"
        "{% elif message['role'] == 'assistant' %}"
        "{{ ' ' + message['content'].strip() + ' </s>' }}"
        "{% endif %}"
        "{% endfor %}"
    )
    print('  chat_template not found — applied LLaMA-2 INST format manually.')

vram_status('MentaLLaMA loaded')
print('✅ MentaLLaMA-chat-7B ready')

In [ ]:
# ================================================================
# CELL 9 — Model 2: MentaLLaMA — Inference on all 3 datasets
# ================================================================
ml_results_list = []

for df, text_col, label_col, ds_name in DATASETS:
    r = run_generative_model(
        df, text_col, label_col,
        ml_model, ml_tokenizer,
        model_name='MentaLLaMA',
        dataset_name=ds_name,
        use_chat_template=True,   # MentaLLaMA uses plain prompt
        max_new_tokens=350
    )
    ml_results_list.append(r)
    vram_status(f'after {ds_name}')

#unload_model(ml_model, ml_tokenizer, 'MentaLLaMA')
print('\n✅ MentaLLaMA complete on all datasets')

In [ ]:
unload_model(ml_model, ml_tokenizer, 'MentaLLaMA')

---
## Model 3 — Mistral-7B-Instruct-v0.2
**Paradigm:** General-purpose LLM guided by structured C-SSRS prompt — no mental health fine-tuning  
**Reference:** Patil et al. (2025) — Mistral achieved lowest ordinal MAE among evaluated models  
**Expected time on RTX 3060:** ~50-65 min for all 3 datasets

In [ ]:
# ================================================================
# CELL 10 — Model 3: Mistral — Load (4-bit quantised)
# Mistral requires chat template applied before tokenisation.
# use_chat_template=True is passed to run_generative_model.
# ================================================================
print('Loading Mistral-7B-Instruct-v0.2 (4-bit NF4 quantisation)...')
print('First run: downloading ~14 GB — may take 10-20 min...')

mistral_tokenizer = AutoTokenizer.from_pretrained(
    'mistralai/Mistral-7B-Instruct-v0.2'
)
mistral_model = AutoModelForCausalLM.from_pretrained(
    'mistralai/Mistral-7B-Instruct-v0.2',
    quantization_config=BNB_4BIT,
    device_map='cuda:0',
    low_cpu_mem_usage=True
)
mistral_model.eval()

vram_status('Mistral loaded')
print('✅ Mistral-7B-Instruct-v0.2 ready')

In [ ]:
# ================================================================
# CELL 11 — Model 3: Mistral — Inference on all 3 datasets
# ================================================================
mistral_results_list = []

for df, text_col, label_col, ds_name in DATASETS:
    r = run_generative_model(
        df, text_col, label_col,
        mistral_model, mistral_tokenizer,
        model_name='Mistral_7B',
        dataset_name=ds_name,
        use_chat_template=True,   # Mistral requires chat template
        max_new_tokens=350
    )
    mistral_results_list.append(r)
    vram_status(f'after {ds_name}')

unload_model(mistral_model, mistral_tokenizer, 'Mistral')
print('\n✅ Mistral complete on all datasets')

In [ ]:
unload_model(mistral_model, mistral_tokenizer, 'Mistral')

---
## Model 4 — Llama-3.1-8B-Instruct
**Paradigm:** Open-source general LLM — on-premise, privacy-compliant, comparable scale to Mistral  
**Reference:** Holley et al. (2025) — Llama-3.1-70B achieved 67.5% on clinical vignettes; Perlis et al. (2025) — Llama-3.1-8B c-stat=0.64 vs GPT-4o 0.67  
**Expected time on RTX 3060:** ~50-65 min for all 3 datasets

In [ ]:
# ================================================================
# CELL 12 — Model 4: Llama-3.1-8B — Load (4-bit quantised)
# Replaces DeepSeek-R1 (restricted by university policy).
# Meta Llama 3.1 license: open for research and commercial use.
# Loaded via Unsloth mirror (no HF gate required).
#   → https://huggingface.co/unsloth/Meta-Llama-3.1-8B-Instruct
# ================================================================
print('Loading Llama-3.1-8B-Instruct (4-bit NF4 quantisation)...')
print('First run: downloading ~15 GB — may take 10-20 min...')
llama_tokenizer = AutoTokenizer.from_pretrained(
    'unsloth/Meta-Llama-3.1-8B-Instruct'
)
llama_model = AutoModelForCausalLM.from_pretrained(
    'unsloth/Meta-Llama-3.1-8B-Instruct',
    quantization_config=BNB_4BIT,
    device_map='cuda:0',
    low_cpu_mem_usage=True
)
llama_model.eval()

vram_status('Llama-3.1-8B loaded')
print('✅ Llama-3.1-8B-Instruct ready')

In [ ]:
# ================================================================
# CELL 13 — Model 4: Llama-3.1-8B — Inference on all 3 datasets
# Llama 3 uses the same chat template system as Mistral.
# ================================================================
llama_results_list = []

for df, text_col, label_col, ds_name in DATASETS:
    r = run_generative_model(
        df, text_col, label_col,
        llama_model, llama_tokenizer,
        model_name='Llama31_8B',
        dataset_name=ds_name,
        use_chat_template=True,   # Llama 3 uses chat template
        max_new_tokens=350
    )
    llama_results_list.append(r)
    vram_status(f'after {ds_name}')

unload_model(llama_model, llama_tokenizer, 'Llama-3.1-8B')
print('\n✅ Llama-3.1-8B complete on all datasets')

In [ ]:
unload_model(llama_model, llama_tokenizer, 'Llama-3.1-8B')

---
## Model 5 — Qwen2.5-7B-Instruct
**Paradigm:** State-of-the-art open-weight general LLM — no mental health fine-tuning, no reasoning distillation  
**Reference:** Qwen Team, Alibaba (2024) — Qwen2.5 Technical Report. Trained on 18T tokens (vs 15T for Llama 3.1).  
**Why included:** Different architecture family from all other models; Apache 2.0, no HuggingFace gate.  
**Expected time on RTX 3060:** ~50-65 min for all 3 datasets

In [ ]:
# ================================================================
# CELL 14 — Model 5: Qwen2.5-7B — Load (4-bit quantised)
# Apache 2.0 licence — no HuggingFace gate, freely downloadable.
# First run downloads ~15 GB from HuggingFace Hub.
# Architecture is distinct from Llama/Mistral (different tokeniser,
# attention design, and training data mix — Alibaba, Sep 2024).
# ================================================================
print('Loading Qwen2.5-7B-Instruct (4-bit NF4 quantisation)...')
print('First run: downloading ~15 GB — may take 10-20 min...')

qwen_tokenizer = AutoTokenizer.from_pretrained(
    'Qwen/Qwen2.5-7B-Instruct'
)
qwen_model = AutoModelForCausalLM.from_pretrained(
    'Qwen/Qwen2.5-7B-Instruct',
    quantization_config=BNB_4BIT,
    device_map='cuda:0',
    low_cpu_mem_usage=True
)
qwen_model.eval()

vram_status('Qwen2.5-7B loaded')
print('✅ Qwen2.5-7B-Instruct ready')

In [ ]:
# ================================================================
# CELL 15 — Model 5: Qwen2.5-7B — Inference on all 3 datasets
# Qwen2.5 uses the same chat template format as Mistral and Llama.
# ================================================================
qwen_results_list = []

for df, text_col, label_col, ds_name in DATASETS:
    r = run_generative_model(
        df, text_col, label_col,
        qwen_model, qwen_tokenizer,
        model_name='Qwen25_7B',
        dataset_name=ds_name,
        use_chat_template=True,
        max_new_tokens=350
    )
    qwen_results_list.append(r)
    vram_status(f'after {ds_name}')

unload_model(qwen_model, qwen_tokenizer, 'Qwen2.5-7B')
print('✅ Qwen2.5-7B complete on all datasets')

---
## Model 6 — IBM Granite 3.2-8B-Instruct
**Paradigm:** Reasoning LLM — generates explicit `<thinking>` chain-of-thought before producing output  
**Reference:** IBM Research (2025) — Granite 3.2 Technical Report. Apache 2.0, no HuggingFace gate.  
**Why included:** Only reasoning model (explicit CoT) from a non-Chinese Western institution at ≤8B scale.  
**Expected time on RTX 3060:** ~50-65 min for all 3 datasets

In [ ]:
# ================================================================
# CELL 16 — Model 6: Granite 3.2-8B — Load (4-bit quantised)
# Apache 2.0 licence — IBM Research, no HuggingFace gate.
# Reasoning model: generates <thinking>...</thinking> chain-of-thought
# before the final JSON answer. parse_llm_response strips these tokens.
# First run downloads ~16 GB from HuggingFace Hub.
# ================================================================
print('Loading IBM Granite 3.2-8B-Instruct (4-bit NF4 quantisation)...')
print('First run: downloading ~16 GB — may take 10-20 min...')

granite_tokenizer = AutoTokenizer.from_pretrained(
    'ibm-granite/granite-3.2-8b-instruct'
)
granite_model = AutoModelForCausalLM.from_pretrained(
    'ibm-granite/granite-3.2-8b-instruct',
    quantization_config=BNB_4BIT,
    device_map='cuda:0',
    low_cpu_mem_usage=True
)
granite_model.eval()

vram_status('Granite 3.2-8B loaded')
print('✅ IBM Granite 3.2-8B-Instruct ready')

In [ ]:
# ================================================================
# CELL 17 — Model 6: Granite 3.2-8B — Inference on all 3 datasets
# thinking=True activates Granite's chain-of-thought reasoning mode.
# The <thinking>...</thinking> block is stripped by parse_llm_response
# before JSON extraction — only the final answer is evaluated.
# ================================================================
granite_results_list = []

for df, text_col, label_col, ds_name in DATASETS:
    cached = load_if_saved('Granite32_8B', ds_name)
    if cached is not None:
        granite_results_list.append(cached)
        continue

    tmp_path = RESULTS_DIR / f'Granite32_8B_{ds_name}_TEMP.xlsx'
    results  = []
    print(f'Granite32_8B on {ds_name} (N={len(df)})...')

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        text       = str(row[text_col])[:2500]
        true_label = int(row[label_col])
        full_text  = f'{SYSTEM_PROMPT}{USER_TEMPLATE.format(text=text)}'

        t0 = time.time()
        try:
            messages = [{'role': 'user', 'content': full_text}]
            # thinking=True activates Granite's explicit reasoning mode
            prompt = granite_tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True,
                thinking=True
            )
            inputs = granite_tokenizer(
                prompt, return_tensors='pt',
                truncation=True, max_length=3500
            ).to(granite_model.device)

            with torch.no_grad():
                out_ids = granite_model.generate(
                    **inputs,
                    max_new_tokens=600,   # extra tokens for <thinking> block
                    do_sample=False,
                    temperature=None,
                    top_p=None,
                    pad_token_id=granite_tokenizer.eos_token_id
                )

            raw    = granite_tokenizer.decode(
                out_ids[0][inputs['input_ids'].shape[1]:],
                skip_special_tokens=True
            ).strip()
            parsed = parse_llm_response(raw)

        except Exception as e:
            parsed = {
                'cssrs_level': -1, 'sak_category': 'Error',
                'confidence': 'none', 'key_signals': [],
                'justification': f'Runtime error: {str(e)[:100]}'
            }

        results.append(
            build_result_row(
                ds_name, 'Granite32_8B', idx,
                true_label, parsed,
                time.time() - t0
            )
        )
        if len(results) % 10 == 0:
            pd.DataFrame(results).to_excel(tmp_path, index=False)

    result_df = pd.DataFrame(results)
    save_to_excel(result_df, 'Granite32_8B', ds_name)
    if tmp_path.exists():
        tmp_path.unlink()
    granite_results_list.append(result_df)
    vram_status(f'after {ds_name}')

unload_model(granite_model, granite_tokenizer, 'Granite32_8B')
print('✅ Granite 3.2-8B complete on all datasets')

---
## Results, Metrics & Visualisations

In [ ]:
# ================================================================
# CELL 14 — Consolidate all results from saved Excel files
# Loads everything from disk so this cell works even if you
# restart the kernel after the inference cells completed.
# ================================================================
import glob

xlsx_files = [
    p for p in RESULTS_DIR.glob('*.xlsx')
    if 'TEMP' not in p.name and 'FINAL' not in p.name and 'METRICS' not in p.name
]

print(f'Found {len(xlsx_files)} result files:')
for f in sorted(xlsx_files):
    print(f'  {f.name}')

all_results = pd.concat(
    [pd.read_excel(f) for f in xlsx_files],
    ignore_index=True
)

# Ensure correct types after loading from Excel
all_results['true_label']  = all_results['true_label'].astype(int)
all_results['pred_label']  = pd.to_numeric(all_results['pred_label'], errors='coerce').fillna(-1).astype(int)
all_results['parse_ok']    = all_results['parse_ok'].astype(bool)
all_results['ordinal_error'] = pd.to_numeric(all_results['ordinal_error'], errors='coerce')

print(f'\nTotal rows loaded: {len(all_results)}')
print()
print('Coverage (rows per model × dataset):')
print(
    all_results.groupby(['model', 'dataset'])
    .size().unstack(fill_value=0)
)

MODEL_ORDER   = ['MentalRoBERTa', 'MentaLLaMA', 'Mistral_7B', 'Llama31_8B', 'Qwen25_7B', 'Granite32_8B']
DATASET_ORDER = ['Reddit_CSSRS', 'HumanAI_Dialogs', '10_Cases', 'MHDialog']
MODEL_COLORS  = {
    'MentalRoBERTa': '#2196F3',
    'MentaLLaMA':    '#FF5722',
    'Mistral_7B':    '#9C27B0',
    'Llama31_8B':    '#FF9800',
    'Qwen25_7B':     '#4CAF50',
    'Granite32_8B':  '#607D8B'
}

In [ ]:
# ================================================================
# CELL 15 — Compute benchmark metrics table
#
# Primary metrics (star) — aligned with key papers in this field:
#
# * Recall HR (>=4)     — clinical safety critical: false negatives
#                          cost lives. Priority metric.
#                          [all papers; clinical deployment standard]
#
# * AUC-ROC HR (binary) — discrimination between high-risk vs not.
#                          C-statistic used by Perlis et al. (2025).
#                          [Gaur et al. 2019; Perlis et al. 2025]
#
# * MAE Ordinal         — magnitude of error on the C-SSRS scale.
#                          Off-by-1 vs off-by-5 treated differently.
#                          [Patil et al. 2025: Mistral MAE=0.4398]
#
# * QWK                 — Quadratic Weighted Kappa. Penalises larger
#                          ordinal errors more. Gold standard for
#                          ordinal agreement in clinical NLP.
#                          [Patil et al. 2025: Mistral QWK=0.8767]
#
# * Spearman rho        — Ordinal rank correlation. Validates that
#                          the model preserves C-SSRS severity ranking.
#                          [Patil et al. 2025: Mistral rho=0.8843]
#
# * Macro F1            — Balanced F1 across all 7 C-SSRS classes.
#                          Standard for imbalanced multi-class NLP.
#                          [all papers; direct comparability]
#
# Secondary metrics (informative, not primary in the papers):
#   Adj Acc (+-1), Specificity HR, F1 HR, Precision HR, Parse Rate
# ================================================================

def compute_metrics(df_all: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for ds in DATASET_ORDER:
        for m in MODEL_ORDER:
            sub   = df_all[
                (df_all['dataset']  == ds) &
                (df_all['model']    == m)  &
                (df_all['parse_ok'] == True)
            ]
            total = len(df_all[
                (df_all['dataset'] == ds) &
                (df_all['model']   == m)
            ])
            if len(sub) == 0:
                continue

            y_true = sub['true_label'].tolist()
            y_pred = sub['pred_label'].tolist()

            # Primary metrics
            mae    = mean_absolute_error(y_true, y_pred)
            f1_mac = f1_score(y_true, y_pred, average='macro', zero_division=0)

            # QWK (Patil et al. 2025)
            try:
                qwk = cohen_kappa_score(y_true, y_pred, weights='quadratic')
            except Exception:
                qwk = float('nan')

            # Spearman rho (Patil et al. 2025)
            try:
                spear_r, _ = spearmanr(y_true, y_pred)
            except Exception:
                spear_r = float('nan')

            # High-risk binary (levels 4-6)
            y_tb    = [1 if l >= 4 else 0 for l in y_true]
            y_pb    = [1 if l >= 4 else 0 for l in y_pred]
            rec_hr  = recall_score(y_tb, y_pb, zero_division=0)
            pre_hr  = precision_score(y_tb, y_pb, zero_division=0)
            f1_hr   = f1_score(y_tb, y_pb, zero_division=0)
            spec_hr = recall_score(y_tb, y_pb, pos_label=0, zero_division=0)

            # AUC-ROC binary high-risk (Gaur et al. 2019; Perlis et al. 2025)
            try:
                auc_hr = roc_auc_score(y_tb, y_pb)
            except Exception:
                auc_hr = float('nan')

            # Secondary metrics
            adj     = np.mean([abs(t-p) <= 1 for t, p in zip(y_true, y_pred)])
            avg_lat = sub['latency_s'].mean() if 'latency_s' in sub else 0

            rows.append({
                'Model':              m,
                'Dataset':            ds,
                'N':                  len(sub),
                'Parse Rate':         f'{len(sub)/total:.0%}',
                # Primary metrics (star)
                'Recall HR (>=4) *':  round(rec_hr,  3),
                'AUC-ROC HR *':       round(auc_hr,  3) if not np.isnan(auc_hr)  else 'N/A',
                'MAE Ordinal *':      round(mae,     3),
                'QWK *':              round(qwk,     3) if not np.isnan(qwk)     else 'N/A',
                'Spearman rho *':     round(spear_r, 3) if not np.isnan(spear_r) else 'N/A',
                'Macro F1 *':         round(f1_mac,  3),
                # Secondary metrics
                'Specificity HR':     round(spec_hr, 3),
                'Precision HR':       round(pre_hr,  3),
                'F1 HR':              round(f1_hr,   3),
                'Adj Acc (+-1)':      round(adj,     3),
                'Avg Latency (s)':    round(avg_lat, 2),
            })
    return pd.DataFrame(rows)


metrics_df = compute_metrics(all_results)

print('=' * 120)
print('BENCHMARK TABLE — LLM-BASED SUICIDE RISK DETECTION (C-SSRS FRAMEWORK)')
print('* = primary metrics aligned with Patil et al. (2025), Gaur et al. (2019), Perlis et al. (2025)')
print('=' * 120)
print(metrics_df.to_string(index=False))

metrics_path = RESULTS_DIR / '_FINAL_metrics_table.xlsx'
with pd.ExcelWriter(metrics_path, engine='xlsxwriter') as writer:
    metrics_df.to_excel(writer, sheet_name='Benchmark Metrics', index=False)
    ws  = writer.sheets['Benchmark Metrics']
    wb  = writer.book
    fmt_primary   = wb.add_format({'bg_color': '#DBEAFE', 'bold': True})
    fmt_secondary = wb.add_format({'bg_color': '#F3F4F6'})
    for col_idx, col_name in enumerate(metrics_df.columns):
        fmt = fmt_primary if '*' in col_name else fmt_secondary
        ws.set_column(col_idx, col_idx, 18, fmt)
print(f'Metrics saved to {metrics_path.name}')

In [ ]:
# ================================================================
# CELL 16 — Figure 1: Primary metrics bar chart (4×4 grid)
# Rows = datasets, Columns = primary metrics
# ================================================================
PRIMARY_METRICS = [
    ('Recall HR (>=4) *', 'Recall\nHigh Risk (≥4)'),
    ('MAE Ordinal *',     'MAE Ordinal\n(lower = better)'),
    ('Macro F1 *',        'Macro F1'),
    ('Adj Acc (+-1)',     'Adjacent Accuracy\n(±1 level)'),
]

fig, axes = plt.subplots(
    len(DATASET_ORDER), len(PRIMARY_METRICS),
    figsize=(16, 10), sharey=False
)
fig.suptitle(
    'Benchmark Results — LLM-based Suicide Risk Detection\n'
    'C-SSRS Framework | SAK-aligned Output | RTX 3060 12 GB',
    fontsize=13, fontweight='bold', y=1.01
)

for r_idx, ds in enumerate(DATASET_ORDER):
    sub = metrics_df[metrics_df['Dataset'] == ds]
    sub = sub.set_index('Model').reindex(MODEL_ORDER).reset_index()

    for c_idx, (metric_col, metric_label) in enumerate(PRIMARY_METRICS):
        ax     = axes[r_idx][c_idx]
        vals   = pd.to_numeric(sub[metric_col], errors='coerce').tolist()
        colors = [MODEL_COLORS.get(m, 'grey') for m in sub['Model']]

        bars = ax.bar(
            range(len(sub)), vals,
            color=colors, alpha=0.85,
            edgecolor='black', linewidth=0.4
        )

        # Invert y-axis for MAE (lower is better); use nanmax to handle NaN rows
        is_mae = 'MAE' in metric_col
        valid  = [v for v in vals if v is not None and not np.isnan(v)]
        y_max  = np.nanmax(valid) if valid else 1.0
        ax.set_ylim(0, y_max * 1.3 + 0.05 if is_mae else 1.1)

        # Clinical target line: Recall HR ≥ 0.80
        if 'Recall' in metric_col:
            ax.axhline(0.80, color='red', lw=0.9, ls='--', alpha=0.7)
            ax.text(len(sub)-0.8, 0.81, 'Target 0.80',
                    fontsize=6.5, color='red')

        # Value annotations (skip NaN — models with 0 valid rows)
        for bar, val in zip(bars, vals):
            if val is not None and not np.isnan(val):
                ax.text(
                    bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.01,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=7
                )

        ax.set_xticks(range(len(sub)))
        ax.set_xticklabels(
            [m.replace('_', '\n') for m in sub['Model'].tolist()],
            fontsize=7
        )
        if c_idx == 0:
            ax.set_ylabel(ds.replace('_', ' '), fontsize=8, fontweight='bold')
        if r_idx == 0:
            ax.set_title(metric_label, fontsize=8.5, fontweight='bold')

# Legend
legend_patches = [
    mpatches.Patch(facecolor=c, label=m)
    for m, c in MODEL_COLORS.items()
]
fig.legend(
    handles=legend_patches, loc='lower center',
    ncol=len(MODEL_COLORS), fontsize=9,
    framealpha=0.9, bbox_to_anchor=(0.5, -0.03)
)

plt.tight_layout()
fig_path = RESULTS_DIR / '_fig1_primary_metrics.png'
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Saved → {fig_path.name}')

In [ ]:
# ================================================================
# CELL 17 — Figure 2: Confusion matrices (4 models × 3 datasets)
# ================================================================
n_m = len(MODEL_ORDER)
n_d = len(DATASET_ORDER)

fig, axes = plt.subplots(n_m, n_d, figsize=(5*n_d, 4.5*n_m))
fig.suptitle(
    'Confusion Matrices — All Models × All Datasets (C-SSRS levels 0-6)',
    fontsize=12, fontweight='bold'
)

for r_idx, m in enumerate(MODEL_ORDER):
    for c_idx, ds in enumerate(DATASET_ORDER):
        ax  = axes[r_idx][c_idx]
        sub = all_results[
            (all_results['model']    == m)  &
            (all_results['dataset']  == ds) &
            (all_results['parse_ok'] == True)
        ]

        if len(sub) == 0:
            ax.set_visible(False)
            continue

        y_true  = sub['true_label'].tolist()
        y_pred  = sub['pred_label'].tolist()
        labels  = sorted(set(y_true + y_pred))
        tnames  = [CSSRS_NAMES.get(l, str(l)) for l in labels]

        cm = confusion_matrix(y_true, y_pred, labels=labels)
        sns.heatmap(
            cm, annot=True, fmt='d', cmap='Blues',
            ax=ax, cbar=False,
            xticklabels=tnames, yticklabels=tnames
        )
        ax.set_title(f'{m}\n{ds}', fontsize=8.5)
        ax.set_xlabel('Predicted', fontsize=8)
        ax.set_ylabel('True',      fontsize=8)
        ax.tick_params(labelsize=7)

plt.tight_layout()
fig_path = RESULTS_DIR / '_fig2_confusion_matrices.png'
fig.savefig(fig_path, dpi=130, bbox_inches='tight')
plt.show()
print(f'💾 Saved → {fig_path.name}')

In [ ]:
# ================================================================
# CELL 18 — Figure 3: Domain shift analysis
# Shows how Macro F1 changes across datasets for each model.
# A drop from Reddit → Dialogues validates domain shift hypothesis.
# ================================================================
shift_rows = []
for m in MODEL_ORDER:
    for ds in DATASET_ORDER:
        sub = metrics_df[(metrics_df['Model'] == m) & (metrics_df['Dataset'] == ds)]
        if len(sub) > 0:
            shift_rows.append({'Model': m, 'Dataset': ds, 'Macro F1': sub['Macro F1 *'].values[0]})

shift_df = pd.DataFrame(shift_rows)

fig, ax = plt.subplots(figsize=(10, 5))
ds_labels = [
    'Reddit C-CSSRS\n(social media)',
    'Human-AI Dialogs\n(multi-turn)',
    '10 Cases\n(clinical PoC)',
    'MHDialog\n(mental health)',
]

for m in MODEL_ORDER:
    sub  = shift_df[shift_df['Model'] == m]
    vals = [
        sub[sub['Dataset'] == ds]['Macro F1'].values[0]
        if len(sub[sub['Dataset'] == ds]) > 0 else np.nan
        for ds in DATASET_ORDER
    ]
    ax.plot(
        ds_labels, vals,
        marker='o', linewidth=2, markersize=8,
        color=MODEL_COLORS[m], label=m
    )
    for x, y in zip(ds_labels, vals):
        if not np.isnan(y):
            ax.annotate(f'{y:.3f}', (x, y),
                        textcoords='offset points', xytext=(5, 5), fontsize=8)

ax.set_xlabel('Dataset (ordered by domain proximity to clinical dialogue)', fontsize=10)
ax.set_ylabel('Macro F1', fontsize=10)
ax.set_title(
    'Domain Shift Analysis\nMacro F1 across datasets '
    '(decrease validates domain shift from social media to clinical dialogue)',
    fontsize=10, fontweight='bold'
)
ax.legend(fontsize=9)
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig_path = RESULTS_DIR / '_fig3_domain_shift.png'
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Saved → {fig_path.name}')

In [ ]:
# ================================================================
# CELL 19 — Qualitative analysis: 10-case PoC
# This section demonstrates explainability of LLMs vs. MentalRoBERTa.
# Include 2-3 of these case analyses in your report.
# ================================================================
cases_res = all_results[all_results['dataset'] == '10_Cases'].copy()

print('=' * 70)
print('QUALITATIVE ANALYSIS — 10-Case Human-AI Dialogs (Assignment PoC)')
print('=' * 70)

for _, case_row in df_10cases.iterrows():
    cid       = case_row['case_id']
    true_lbl  = case_row['true_label']
    true_str  = case_row['true_label_str']
    true_sak  = cssrs_to_sak(true_lbl)

    print(f'\n{"─"*65}')
    print(f'CASE {cid}')
    print(f'  True label:  C-SSRS {true_lbl} — {true_str} ({true_sak})')
    print(f'  Text (first 200 chars): {str(case_row["text"])[:200]}...')
    print()

    preds = cases_res[cases_res['idx'] == case_row.name]

    for _, p in preds.iterrows():
        pred    = p['pred_label']
        err     = p['ordinal_error']
        fn_flag = ' ⚠️ FALSE NEGATIVE (HIGH RISK MISSED)' \
                  if (true_lbl >= 4 and pred < 4) else ''
        fp_flag = ' ℹ️ FALSE POSITIVE' \
                  if (true_lbl < 4 and pred >= 4) else ''

        print(f'  [{p["model"]:<18}] pred=C-SSRS {pred} ({p["pred_sak"]}) '
              f'| err=±{err} | conf={p["confidence"]}'
              f'{fn_flag}{fp_flag}')

        # For generative models: show explanation (key differentiator vs. encoder)
        if p['model'] != 'MentalRoBERTa' and str(p['key_signals']) not in ('[]', 'nan', ''):
            print(f'    Signals:    {p["key_signals"]}')
        if p['model'] != 'MentalRoBERTa':
            print(f'    Justif.:   {str(p["justification"])[:220]}')

# ── Summary: false negatives on high-risk cases ──────────────
print(f'\n{"="*70}')
print('HIGH-RISK FALSE NEGATIVE SUMMARY (clinical safety critical)')
hr_rows = cases_res[cases_res['hr_true'] == 1]
fn_rows = hr_rows[hr_rows['hr_pred'] == 0]
print(f'High-risk cases (true label ≥4): {len(hr_rows)}')
print(f'False negatives detected:        {len(fn_rows)}')
if len(fn_rows) > 0:
    print(fn_rows[['model', 'idx', 'true_label', 'pred_label', 'justification']].to_string(index=False))

In [ ]:
# ================================================================
# CELL 20 — Recommended model selection
#
# Composite clinical score:
#   Score = F1_HR - (MAE / 6) + (Macro_F1 / 2)
#
# Why F1_HR instead of raw Recall_HR:
#   Recall HR alone rewards any model that says "high risk" for
#   every case — a trivial strategy that is clinically useless.
#   F1_HR is the harmonic mean of Recall_HR and Precision_HR:
#   it penalises false alarms (precision) equally to missed cases
#   (recall), so the "predict everything high risk" strategy scores
#   low on Precision_HR and therefore low on F1_HR.
#   MAE further penalises models that are far off on the ordinal scale.
#   Macro F1 rewards models that can distinguish all seven C-SSRS levels.
# ================================================================
print('MODEL RECOMMENDATION — Composite Clinical Score')
print('Score = F1_HR - (MAE_Ordinal/6) + (Macro_F1/2)')
print('F1_HR balances recall (catch all high-risk) with precision (avoid false alarms)\n')

avg_metrics = (
    metrics_df
    .groupby('Model')[['F1 HR', 'Recall HR (>=4) *', 'MAE Ordinal *', 'Macro F1 *']]
    .mean()
    .round(3)
)
avg_metrics['Composite Score'] = (
    avg_metrics['F1 HR']
    - avg_metrics['MAE Ordinal *'] / 6
    + avg_metrics['Macro F1 *'] / 2
).round(3)

ranking = avg_metrics.sort_values('Composite Score', ascending=False)
print(ranking.to_string())

best_model = ranking.index[0]
print(f'\n⭐ Recommended model: {best_model}')
print(
    'Justification: Highest composite clinical score across all 4 datasets. '
    'F1_HR prevents trivially high recall from dominating; MAE penalises '
    'large ordinal errors; Macro F1 rewards multi-class discrimination.'
)

# Save recommendation to Excel
rec_path = RESULTS_DIR / '_FINAL_model_recommendation.xlsx'
ranking.to_excel(rec_path)
print(f'\n💾 Saved → {rec_path.name}')

In [ ]:
# ================================================================
# CELL 21 — Final output summary
# Lists all files generated for the report.
# ================================================================
print('ALL FILES GENERATED FOR YOUR REPORT')
print('=' * 60)

all_output = sorted(RESULTS_DIR.iterdir())
xlsx_files = [f for f in all_output if f.suffix == '.xlsx']
png_files  = [f for f in all_output if f.suffix == '.png']

print('\nExcel results (model × dataset — raw predictions):')
for f in xlsx_files:
    if 'FINAL' not in f.name and 'METRICS' not in f.name:
        print(f'  {f.name:<55} {f.stat().st_size/1024:.0f} KB')

print('\nExcel summary files:')
for f in xlsx_files:
    if 'FINAL' in f.name or 'METRICS' in f.name:
        print(f'  {f.name:<55} {f.stat().st_size/1024:.0f} KB')

print('\nFigures (PNG):')
for f in png_files:
    print(f'  {f.name:<55} {f.stat().st_size/1024:.0f} KB')

print(f'\n✅ Benchmark complete.')
print(f'   Results directory: {RESULTS_DIR.resolve()}')